[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/spin_transformations/spin_transformations.ipynb)

# Spin Transformations

This example computes conformal surface deformations that change mean curvature by a prescribed amount while preserving every angle on the surface. A turn and a scale at each vertex act on the edges around it, and repeating the step drives conformal curvature flow that rounds Spot the cow into a sphere, whereas prescribing a constant curvature shift yields the spinor spherical harmonics and spins a sphere into Dirac spheres.

In three-dimensional space the turn and scale at each vertex is modelled as a scaled rotor, an even multivector, while vertices and edge vectors are modelled as intrinsic points, antivectors. The Dirac operator couples each triangle to its corners through the opposite edge vectors, taking even vertex fields to even face fields, and minimizing the area-weighted squared residual `~residual * face_areas * residual` yields the rotor field before sandwiching each old edge via `rotors << edges`.

In the quaternion matrix notation of [Crane, Pinkall and Schröder (2011)](https://www.cs.cmu.edu/~kmcrane/Projects/SpinTransformations/paper.pdf), imaginary quaternions $\mathrm{Im}(\mathbb{H}) \cong \mathbb{R}^3$ hold both vertex coordinates and edge vectors, the discrete Dirac matrix has off-diagonal quaternion entries $D_{ij} = -\tfrac{1}{2\mathcal{A}_i} e_j$ shifted by real scalar entries $R$ into $A = D - R$, the smallest generalized eigenvalue problem $A^H M_F A\,\lambda = \gamma\, M_V \lambda$ over $4|V|$ real components selects $\lambda \in \mathbb{H}^{|V|}$, and each edge updates via the quaternion product $\tilde e_{ij} = \bar\lambda\, e_{ij}\, \lambda$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology

from examples.surfaces.spin_transformations import render
from examples.surfaces.spin_transformations.scenarios import spot
from examples.util import shapes
from examples.util.animation import save_figure, show_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Plane = ga.gatype.vector()                                                     # Vector+
# Vertices and edge vectors are intrinsic points (antivectors); vertex turns and scales are even multivectors.
Point = ga.gatype.antivector()                                                 # Antivector-
Patch = ga.gatype.antibivector()                                               # Antibivector-
Even = ga.gatype.even()                                                        # Even+
# Joining three points gives an intrinsic volume scalar, whose unit carries that orientation onto ordinary scalars.
unit_volume = mv.xyz.dual()                                                    # [] Antipseudoscalar-

np.set_printoptions(precision=4, suppress=True)

## Operators between fields on a triangulated surface

A field attaches one algebraic element to each vertex, edge or face of a triangulated surface, and a linear operator between fields couples incident sites by multivector multiplication while `~` reverses both the coupling direction and every multivector cell. The incidence operators couple sites by plain numbers, so `boundary * vertices` forms the oriented edge vectors and `boundary.adjugate() * edge_weights * boundary` assembles the cotangent Laplacian.

In [ ]:
# Subdivisions of the icosahedron forming the unit sphere.
n_subdivisions = 3

In [ ]:
topology, coordinates = shapes.sphere(n_subdivisions)
vertices = mv.vector(coordinates).dual().field()                               # Point-[V]
print(vertices.gatype, topology.elements[1].shape, topology.elements[2].shape)

In [ ]:
def cotangent_weights(topology: Topology, sides: Point) -> Scalar:
    """Each edge's cotangent weight (`Scalar+[E]`) is half the sum of the cotangents of the two angles
    facing it, read from the edge vectors `sides` (`[3] Point-[F]`) facing each corner."""
    tails, heads = topology.corners[2, 1]
    after, before = sides[tails], sides[heads]                                 # [3] Point-[F], [3] Point-[F]
    cotangents = (after | before) / (after & before).norm()                    # [3] Scalar+[F]
    return (topology.incidence[2, 1].adjugate() * (cotangents / 2)).sum(axis=0)  # Scalar+[E]

In [ ]:
# Boundary difference along each edge, and incidence averages onto edges and faces.
boundary = topology.d[0]                                                       # Scalar+[E <- V]
edge_means = topology.incidence[1, 0].mean(axis=0)                             # Scalar+[E <- V]
face_means = topology.incidence[2, 0].mean(axis=0)                             # Scalar+[F <- V]

edges = boundary * vertices                                                    # Point-[E]
# Each corner of a face sees the opposite edge signed along the face boundary.
sides = topology.boundary[2, 1] * edges                                        # [3] Point-[F]
# Joining two sides gives twice the oriented triangle patch, whose norm is twice the area.
patches = (sides[0] & sides[1]) / 2                                            # Patch-[F]
areas = patches.norm()                                                         # Scalar+[F]
# Diagonal mass maps on faces and vertices, assigning one third of each triangle's area to each corner.
face_areas, vertex_areas = spdiag(areas), spdiag(face_means.adjugate() * areas)  # Scalar+[F <- F], Scalar+[V <- V]
edge_weights = spdiag(cotangent_weights(topology, sides))                      # Scalar+[E <- E]
laplacian = boundary.adjugate() * edge_weights * boundary                      # Scalar+[V <- V]

print((boundary * Scalar).gatype, boundary.shape)
print(edges.gatype)

## Prescribing a change in mean curvature

A scalar field on the faces prescribes how much the mean curvature changes across the surface. Pairing the coordinate plane `y` with each face centroid `y & centroids` builds a dipole profile that asks the sphere to bend more on one side and flatten on the opposite side.

In [ ]:
# Strength of the linear mean-curvature gradient along y.
dipole = 0.5

centroids = face_means * vertices                                              # Point-[F]
curvature_change = (mv.y & centroids) * dipole                                 # Scalar+[F]

figure = render.surface_figure(topology, vertices, curvature_change)
save_figure(figure, "spin_transformations_curvature")
show_figure(figure);

Red faces carry positive prescribed curvature change and blue faces carry negative change.

## The Dirac operator and simplex gradients

The term ***simplex gradient*** here means the vector on each triangle that pairs with displacements in the tangent plane to give the linear change of a corner's value across the triangle, `(sides | patches.inverse()) / 2`. Multiplying the piecewise-linear geometric derivative on the left by the negative unit face normal turns each corner's simplex gradient a quarter turn within the triangle into `sides / (2 * areas)`, recovering the discrete Dirac operator that maps even vertex fields to even face fields. Shifting each corner coupling by the prescribed curvature change in `unit_volume` aligns the scalar shift's intrinsic orientation with the edge bivectors, so the squared residual `~residual * face_areas * residual` forms a symmetric even operator on the vertices.

In finite-element and quaternion matrix notation, the simplex gradients are the gradients of the piecewise-linear hat functions $\nabla \phi_j$, and the shifted Dirac residual writes $A = D - R$ with quaternion matrix entries $D_{ij} = -\tfrac{1}{2\mathcal{A}_i} e_j$ under Hamilton's sign convention and Hermitian energy matrix $A^H M_F A$.

In [ ]:
def geometric_derivative(topology: Topology, vertices: Point) -> SparseExtensor:
    """The geometric derivative (`Plane+[F <- V]`) of piecewise-linear vertex fields couples each face
    to each of its corners through that corner's simplex gradient, the opposite edge contracted with
    the inverse of twice the face's oriented patch."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)               # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                        # Patch-[F]
    gradients = (sides | patches.inverse()) / 2                                # [3] Plane+[F]
    return (spdiag(gradients) * topology.incidence[2, 0]).sum(axis=0)          # Plane+[F <- V]


dirac = (spdiag(sides / (2 * areas)) * topology.incidence[2, 0]).sum(axis=0)   # Point-[F <- V]
face_normals = patches / areas                                                 # Patch-[F]
derivative = geometric_derivative(topology, vertices)                          # Plane+[F <- V]
# Shifting the Dirac couplings by one third of the face's curvature change at each corner.
residual = (spdiag(sides / (2 * areas) - curvature_change * unit_volume / 3) * topology.incidence[2, 0]).sum(axis=0)  # Even-[F <- V]
energy = ~residual * face_areas * residual                                     # Even+[V <- V]

print(derivative.gatype, dirac.gatype)
print(energy.gatype, energy.shape)

## Solving for the rotor field and new surface

Restricting `energy * Even` and `vertex_areas * Even` to maps on even multivectors turns the least-residual search into a generalized eigensolve per unit vertex area, normalized by its area-weighted mean to fix global rotation and scale. Integrating the reverse sandwich `rotor << edge` along each edge by Simpson's rule averages the endpoint quadratic maps `rotors << Point` independently of rotor sign alongside twice the midpoint sandwich, and a cotangent-weighted Poisson solve reconstructs the deformed vertex positions.

In quaternion notation, the integrated edge update writes $\tilde e_{ij} = \tfrac13 \bar\lambda_i e_{ij} \lambda_i + \tfrac16 \bar\lambda_i e_{ij} \lambda_j + \tfrac16 \bar\lambda_j e_{ij} \lambda_i + \tfrac13 \bar\lambda_j e_{ij} \lambda_j$.

In [ ]:
# Least eigenfield of the residual energy per unit vertex area, normalized by its area-weighted mean.
_, (field,) = (energy * Even).eigh(vertex_areas * Even, 1)                     # Even+[V]
rotors = field / ((vertex_areas * field).sites.sum() / areas.sites.sum())      # Even+[V]
scaling = (face_means * rotors).scalar_norm_squared() - 1                      # Scalar+[F]

figure = render.surface_figure(topology, vertices, scaling)
save_figure(figure, "spin_transformations_scaling")
show_figure(figure);

Red faces expand under the mean vertex rotor and blue faces contract.

In [ ]:
def deformed_surface(topology: Topology, vertices: Point, curvature_change: Scalar) -> Point:
    """The spin transformation for a prescribed `curvature_change` (`Scalar+[F]`) turns the edges by the
    rotor field of least Dirac residual energy per unit vertex area and solves for the vertices (`Point-[V]`)
    whose edges best match the turned ones."""
    edge_means = topology.incidence[1, 0].mean(axis=0)                         # Scalar+[E <- V]
    face_means = topology.incidence[2, 0].mean(axis=0)                         # Scalar+[F <- V]
    edges = topology.d[0] * vertices                                           # Point-[E]
    sides = topology.boundary[2, 1] * edges                                    # [3] Point-[F]
    areas = (sides[0] & sides[1]).norm() / 2                                   # Scalar+[F]
    face_areas, vertex_areas = spdiag(areas), spdiag(face_means.adjugate() * areas)  # Scalar+[F <- F], Scalar+[V <- V]
    edge_weights = spdiag(cotangent_weights(topology, sides))                  # Scalar+[E <- E]
    laplacian = topology.d[0].adjugate() * edge_weights * topology.d[0]        # Scalar+[V <- V]

    residual = (spdiag(sides / (2 * areas) - curvature_change * unit_volume / 3) * topology.incidence[2, 0]).sum(axis=0)  # Even-[F <- V]
    _, (field,) = ((~residual * face_areas * residual) * Even).eigh(vertex_areas * Even, 1)  # Even+[V]
    rotors = field / ((vertex_areas * field).sites.sum() / areas.sites.sum())  # Even+[V]
    turned_edges = ((edge_means * (rotors << Point))(edges) + 2 * ((edge_means * rotors) << edges)) / 3  # Point-[E]
    return (laplacian * Point).lstsq(topology.d[0].adjugate() * edge_weights * turned_edges)  # Point-[V]


transformed_vertices = deformed_surface(topology, vertices, curvature_change)  # Point-[V]

figure = render.textured_figure(topology, transformed_vertices, vertices)
save_figure(figure, "spin_transformations_deformed")
show_figure(figure);

Carrying the sphere's 3D checkerboard onto the pear-shaped surface preserves right-angled corners across the squares because the spin transformation is conformal.

## Conformal curvature flow

Removing a fixed share of each face's departure from the area-weighted mean curvature at every step relaxes a surface toward constant mean curvature while preserving corner angles throughout. Streaming sixty steps on Spot the cow takes roughly half a minute.

In [ ]:
# Share of curvature departure removed per step, step count, and frame duration.
rate = 0.05
n_steps = 60
duration_ms = 40

In [ ]:
def mean_curvature(topology: Topology, vertices: Point) -> Scalar:
    """Each face's signed mean curvature (`Scalar+[F]`) averages the vertex mean curvatures, where the
    cotangent Laplacian of the positions gives the area-integrated mean-curvature normal met with the
    unit vertex normal over twice the vertex area."""
    face_means = topology.incidence[2, 0].mean(axis=0)                         # Scalar+[F <- V]
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)               # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                        # Patch-[F]
    areas = patches.norm()                                                     # Scalar+[F]
    laplacian = topology.d[0].adjugate() * spdiag(cotangent_weights(topology, sides)) * topology.d[0]  # Scalar+[V <- V]
    # Summed face patches around each vertex give an area-weighted normal once normalized.
    normals = (face_means.adjugate() * patches).normalized()                   # Patch-[V]
    # Meeting the integrated mean-curvature normal with the vertex normal yields an intrinsic volume scalar.
    integrated_curvature = ((laplacian * vertices) & normals) * unit_volume    # Scalar+[V]
    return face_means * (integrated_curvature / (2 * (face_means.adjugate() * areas)))  # Scalar+[F]


def recentred_vertices(vertices: Point) -> Point:
    """Centring `vertices` (`Point-[V]`) and scaling to unit mean radius fixes the translation and
    overall scale that a spin transformation leaves free."""
    centred = vertices - vertices.sites.mean()                                 # Point-[V]
    return centred / centred.norm().sites.mean()                               # Point-[V]

In [ ]:
def curvature_flow_states(topology: Topology, vertices: Point, n_steps: int, rate: float) -> Iterator[Point]:
    """Yields the surface vertices (`Point-[V]`) at the start and after each of `n_steps` steps of
    conformal curvature flow, where each step removes a share `rate` of every face's departure from
    the area-weighted mean curvature."""
    vertices = recentred_vertices(vertices)                                    # Point-[V]
    yield vertices
    for _ in range(n_steps):
        curvature = mean_curvature(topology, vertices)                         # Scalar+[F]
        sides = topology.boundary[2, 1] * (topology.d[0] * vertices)           # [3] Point-[F]
        areas = (sides[0] & sides[1]).norm() / 2                               # Scalar+[F]
        mean = (curvature * areas).sites.sum() / areas.sites.sum()             # [] Scalar+
        vertices = recentred_vertices(deformed_surface(topology, vertices, -rate * (curvature - mean)))  # Point-[V]
        yield vertices


cow_topology, cow_vertices = spot()
cow_frames = show_states(
    curvature_flow_states(cow_topology, cow_vertices, n_steps, rate),
    render.rounding_animation,
    cow_topology,
    name="spin_rounding",
    duration_ms=duration_ms,
);

Spot draws in her legs, horns and ears into a sphere while the checkerboard squares mapped from the initial cow keep their right angles.

## Dirac spheres

When the prescribed curvature shift is a constant integer other than `-1`, the zero-energy fields of the shifted residual are eigenfields of the spherical Dirac operator itself, called spinor spherical harmonics, and spinning a finely subdivided unit sphere by them produces immersed constant-mean-curvature Dirac spheres.

In [ ]:
# Subdivisions of the finer sphere, integer Dirac eigenvalues, and eigenmodes per row.
n_dirac_subdivisions = 4
eigenvalues = (1, 2, 3, -2, -3)
n_states = 3

In [ ]:
def dirac_spheres(topology: Topology, vertices: Point, eigenvalues: np.ndarray, n_states: int) -> tuple[Scalar, Point]:
    """Each constant curvature shift in `eigenvalues` (`[n_eigenvalues]`) gives the `n_states` rotor
    eigenfields of least Dirac residual energy (`[n_eigenvalues, n_states] Scalar+`) and spins the
    surface into the corresponding Dirac spheres (`[n_eigenvalues, n_states] Point-[V]`)."""
    edge_means = topology.incidence[1, 0].mean(axis=0)                         # Scalar+[E <- V]
    face_means = topology.incidence[2, 0].mean(axis=0)                         # Scalar+[F <- V]
    edges = topology.d[0] * vertices                                           # Point-[E]
    sides = topology.boundary[2, 1] * edges                                    # [3] Point-[F]
    areas = (sides[0] & sides[1]).norm() / 2                                   # Scalar+[F]
    face_areas, vertex_areas = spdiag(areas), spdiag(face_means.adjugate() * areas)  # Scalar+[F <- F], Scalar+[V <- V]
    edge_weights = spdiag(cotangent_weights(topology, sides))                  # Scalar+[E <- E]
    laplacian = topology.d[0].adjugate() * edge_weights * topology.d[0]        # Scalar+[V <- V]

    shifts = eigenvalues[:, None] * unit_volume / 3                            # [..., 1] Antipseudoscalar-
    residual = (spdiag(sides / (2 * areas) - shifts) * topology.incidence[2, 0]).sum(axis=-1)  # [...] Even-[F <- V]
    energies, eigenfields = ((~residual * face_areas * residual) * Even).eigh(vertex_areas * Even, n_states)  # [..., n_states] Even+[V]
    turned_edges = ((edge_means * (eigenfields << Point))(edges) + 2 * ((edge_means * eigenfields) << edges)) / 3  # [..., n_states] Point-[E]
    return energies, (laplacian * Point).lstsq(topology.d[0].adjugate() * edge_weights * turned_edges)


fine_topology, fine_coordinates = shapes.sphere(n_dirac_subdivisions)
fine_vertices = mv.vector(fine_coordinates).dual().field()                     # Point-[V]
dirac_energies, dirac_vertices = dirac_spheres(fine_topology, fine_vertices, np.array(eigenvalues, dtype=float), n_states)  # [..., n_states] Scalar+, [..., n_states] Point-[V]

figure = render.dirac_figure(fine_topology, dict(zip(eigenvalues, dirac_vertices)), fine_vertices)
save_figure(figure, "dirac_spheres")
show_figure(figure);

Warm orange shades mark the outer side of the reference sphere and cool blue shades mark the inner side where higher-frequency modes turn the immersed surface through itself.

In [ ]:
# checks
def corner_cosines(topology: Topology, vertices: Point) -> Scalar:
    """Each triangle corner's angle cosine (`[3] Scalar+[F]`) comes from the inner product of the two
    edges meeting there over the product of their lengths."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)               # [3] Point-[F]
    tails, heads = topology.corners[2, 1]
    after, before = sides[tails], sides[heads]                                 # [3] Point-[F], [3] Point-[F]
    return (after | before) / (after.norm() * before.norm())                   # [3] Scalar+[F]


def surface_roundness(topology: Topology, vertices: Point) -> Scalar:
    """The relative spread (`[] Scalar+`) of vertex distances from the surface's area-weighted centroid,
    which vanishes on any sphere regardless of vertex crowding."""
    centroids = topology.incidence[2, 0].mean(axis=0) * vertices               # Point-[F]
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)               # [3] Point-[F]
    areas = (sides[0] & sides[1]).norm() / 2                                   # Scalar+[F]
    middle = (centroids * areas).sites.sum() / areas.sites.sum()               # [] Point-
    distances = (vertices - middle).norm()                                     # Scalar+[V]
    mean_distance = distances.sites.mean()                                     # [] Scalar+
    return (distances - mean_distance).squared().sites.mean().square_root() / mean_distance  # [] Scalar+


# Crane's Dirac operator equals the geometric derivative multiplied on the left by the negative unit face normal,
# and the geometric derivative of position equals twice the intrinsic pseudoscalar on every face.
operator_difference = spdiag(face_normals) * derivative + dirac                # Even-[F <- V]
assert ((~operator_difference * operator_difference).diagonal().scalar_norm_squared() < 1e-24).all()
assert (((derivative * vertices).dual() - 2).scalar_norm_squared() < 1e-24).all()

# Dipole deformation preserves triangle corner angles up to surface resolution.
assert ((corner_cosines(topology, transformed_vertices) - corner_cosines(topology, vertices)).abs() < 0.05).all()

# Conformal curvature flow tightens Spot's radial spread around her area-weighted centroid tenfold while preserving mean corner cosines.
assert surface_roundness(cow_topology, cow_frames[-1]) < 0.1 * surface_roundness(cow_topology, cow_frames[0])
assert (corner_cosines(cow_topology, cow_frames[-1]) - corner_cosines(cow_topology, cow_frames[0])).abs().sites.mean().mean() < 0.05

# Integer Dirac eigenvalues achieve near-zero residual energy.
assert (dirac_energies.abs() < 0.05).all()